# Step 1 — Font Re-Rendering (PIYA-106 BUG-011 retraining plan)

Implements step 1 of `docs/decisions/piya-106-crnn-font-retraining-plan.md`. Re-renders every
CameraPrIMuS `.mei` file through 4 real music-engraving fonts (Bravura, Petaluma, Gootville, Leland)
via Verovio, producing new training images whose existing `.agnostic`/`.semantic` labels stay valid
unchanged (only the glyph style changes, not the music content) — zero manual transcription needed.

Full writeup / rationale / sanity-check steps: `training/step1-font-rerender-kaggle-guide.md`.

**Kaggle-native**: assumes the `piamac/cameraprimus` dataset is already attached to this notebook via
**Add Data** (same dataset the cells below already read from at
`/kaggle/input/datasets/piamac/cameraprimus/CameraPrIMuS - Copy`) — no download step needed. Output
goes to `/kaggle/working/cpm_font_augmented`; commit this notebook (Save Version → Save & Run All) and
then use **+ New Dataset** on the output to persist it as its own Kaggle dataset for step 2.

**Run this section once.** It only produces the new font-varied images — it does **not** wire them
into `CameraPrIMuSDataset`/`train()` below. That's step 2 (fine-tuning), which is intentionally not
started here: `docs/decisions/piya-106-crnn-font-retraining-plan.md` and CLAUDE.md's Agent Usage Rules
both require a fresh ml-cv-lead review before any fine-tuning code is written against this output.

In [ ]:
# Step 1, Cell A — system + Python deps for SVG rasterization
# Gated off by default (2026-09-23): Step 1's output is already persisted as the
# piamac/cpm-font-augmented2 Kaggle input dataset that Step 2 Cell B reads from --
# re-running Step 1 on a fresh session just burns time re-rendering the whole corpus
# for output nothing downstream consumes. Flip to True only to add/change a font.
RUN_STEP1_FONT_RENDER = False
if RUN_STEP1_FONT_RENDER:
    !apt-get -qq update && apt-get -qq install -y libcairo2
    !pip install -q verovio cairosvg

In [ ]:
if RUN_STEP1_FONT_RENDER:
    # Step 1, Cell B — render every CameraPrIMuS sample's .mei through each font.
    # Reads the SAME dataset already attached above (piamac/cameraprimus, via Kaggle
    # "Add Data" -- no kagglehub download needed on Kaggle). Resumable: re-running
    # this cell skips any (stem, font) pair whose PNG already exists in OUT_DIR.

    import verovio, cairosvg, os, glob, shutil, re
    from pathlib import Path

    CPM_ROOT   = Path("/kaggle/input/datasets/piamac/cameraprimus/CameraPrIMuS - Copy")
    CORPUS_DIR = CPM_ROOT / "Corpus"
    OUT_DIR    = Path("/kaggle/working/cpm_font_augmented2")
    FONTS      = ["Bravura", "Petaluma", "Gootville", "Leland"]

    RENDER_OPTS = {
        "inputFrom": "mei",
        "breaks": "none",
        "adjustPageHeight": True,
        "adjustPageWidth": True,
        "header": "none",
        "footer": "none",
        "scale": 100,
        "pageMarginTop": 0,
        "pageMarginBottom": 0,
        "pageMarginLeft": 0,
        "pageMarginRight": 0,
    }

    mei_files = sorted(CORPUS_DIR.rglob("*.mei"))
    print(f"Found {len(mei_files):,} .mei files")

    failures = []
    for i, mei_path in enumerate(mei_files):
        stem = mei_path.stem
        sample_dir = mei_path.parent
        agnostic_src = sample_dir / f"{stem}.agnostic"
        semantic_src = sample_dir / f"{stem}.semantic"
        if not agnostic_src.exists():
            continue  # skip anything without ground truth to pair against

        for font in FONTS:
            out_sample_dir = OUT_DIR / font / stem
            out_sample_dir.mkdir(parents=True, exist_ok=True)
            png_path = out_sample_dir / f"{stem}.png"
            if png_path.exists():
                continue  # resumable -- skip work already done in a prior run

            try:
                tk = verovio.toolkit()
                tk.setOptions({**RENDER_OPTS, "font": font})
                if not tk.loadFile(str(mei_path)):
                    failures.append((stem, font, "loadFile returned False"))
                    continue
                svg = tk.renderToSVG(1)
                svg_path = out_sample_dir / f"{stem}.svg"
                svg_path.write_text(svg, encoding="utf-8")
                cairosvg.svg2png(url=str(svg_path), write_to=str(png_path),
                                  background_color="white", scale=2.0)
                svg_path.unlink()  # keep only the PNG once rasterized

                # copy the ORIGINAL, unmodified labels next to the new render --
                # the music content is identical, so these are still correct ground truth
                shutil.copy(agnostic_src, out_sample_dir / f"{stem}.agnostic")
                if semantic_src.exists():
                    shutil.copy(semantic_src, out_sample_dir / f"{stem}.semantic")
            except Exception as e:
                failures.append((stem, font, str(e)))

        if (i + 1) % 2000 == 0:
            print(f"  [{i+1:,} / {len(mei_files):,}]")

    print(f"\nDone. {len(failures)} failures out of {len(mei_files) * len(FONTS):,} attempted renders.")
    for stem, font, err in failures[:20]:
        print(f"  FAILED {stem} / {font}: {err}")

## Step 1 sanity checks (do before trusting the output)

1. **Visually spot-check ~10 samples per font** — open a few new PNGs under
   `/kaggle/working/cpm_font_augmented2/<font>/<stem>/` next to the original CameraPrIMuS `.png` for
   the same stem. Confirm: single staff line, correct clef/notes, no clipped noteheads/stems, no
   leftover title/composer text.
2. **Confirm label parity** — for a few samples, re-open the copied `.agnostic` and manually count
   clef/note/barline tokens against what's visible in the new PNG; they must match exactly.
3. **Check file counts** — `len(mei_files) * len(FONTS)` PNGs expected minus any in `failures`. A
   large failure count concentrated on one font (not scattered across all) suggests a font-specific
   glyph-coverage gap worth investigating before using that font's output.
4. **Persist the output** — `/kaggle/working` survives only until this session/commit is cleaned up.
   Commit this notebook (**Save Version → Save & Run All (Commit)**), then on the Output tab of the
   finished run use **+ New Dataset** to turn `cpm_font_augmented2` into its own Kaggle dataset you can
   attach as input once step 2 (fine-tuning) is ready to consume it.

### What this does NOT yet do

- Does not touch `CameraPrIMuSDataset` or `train()` above — that's step 2, not started. Per
  `docs/decisions/piya-106-crnn-font-retraining-plan.md` and CLAUDE.md's Agent Usage Rules, step 2
  needs a fresh ml-cv-lead review before any fine-tuning code is written.
- Condition A only (bare re-render) — no realistic strip context (label text/dynamics clutter)
  composited in yet; ml-cv-lead's review flagged that as a separate follow-up, not part of this step.
- No distorted/camera-simulation pass over the new font renders — `CameraPrIMuSDataset._augment()`
  above can likely be reused as-is once these renders are wired into the loader in step 2.

In [ ]:
import collections
from pathlib import Path
from tqdm.notebook import tqdm  # ← notebook version looks nicer in Kaggle

def build_vocab_from_dataset(root_dir, min_count=2):
    files = list(Path(root_dir).rglob("*.agnostic"))
    print(f"Found {len(files):,} .agnostic files. Starting scan...\n")

    counter = collections.Counter()
    for i, path in enumerate(tqdm(files, desc="Building vocab", unit="file")):
        with open(path, encoding="utf-8", errors="ignore") as f:
            counter.update(f.read().split())

        if (i + 1) % 5000 == 0:
            print(f"  [{i+1:,} / {len(files):,}] — {len(counter):,} unique tokens so far")

    vocab = ["<blank>", "<unk>"]
    for token, count in counter.most_common():
        if count >= min_count:
            vocab.append(token)

    print(f"\nVocab size: {len(vocab)} tokens")
    print(f"Dropped (count=1): {sum(1 for c in counter.values() if c < min_count)} tokens")
    return vocab, counter

vocab, counter = build_vocab_from_dataset("/kaggle/input/datasets/piamac/cameraprimus/CameraPrIMuS - Copy")

print("\nVOCAB = [")
for t in vocab:
    print(f'    "{t}",')
print("]")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# ── Vocabulary ────────────────────────────────────────────────────────────────
VOCAB = [
    "<blank>",
    "<unk>",
    "barline-L1",
    "clef.G-L2",
    "note.quarter-L4",
    "rest.eighth-L3",
    "note.quarter-S3",
    "rest.quarter-L3",
    "note.quarter-L3",
    "note.quarter-S4",
    "metersign.C-L3",
    "note.quarter-S2",
    "dot-S3",
    "note.eighth-L4",
    "dot-S4",
    "note.eighth-S3",
    "note.eighth-S4",
    "note.quarter-L5",
    "note.quarter-L2",
    "note.eighth-L3",
    "accidental.flat-L3",
    "note.eighth-S2",
    "accidental.sharp-L5",
    "multirest-L3",
    "clef.C-L1",
    "digit.4-L2",
    "dot-S2",
    "note.eighth-L5",
    "digit.3-L4",
    "note.eighth-L2",
    "note.quarter-S5",
    "dot-S5",
    "metersign.C/-L3",
    "rest.half-L3",
    "note.half-L4",
    "accidental.sharp-S3",
    "note.half-S3",
    "note.quarter-S1",
    "accidental.flat-S4",
    "note.beamedRight1-L4",
    "note.eighth-S5",
    "note.half-L3",
    "note.beamedRight1-S3",
    "note.half-S4",
    "note.half-S2",
    "note.quarter-L6",
    "note.beamedRight1-L3",
    "note.beamedRight1-S4",
    "accidental.flat-L4",
    "digit.1-S5",
    "accidental.sharp-S2",
    "note.beamedLeft2-L4",
    "note.beamedLeft2-S3",
    "note.beamedBoth2-L4",
    "digit.2-L4",
    "note.eighth-S1",
    "note.quarter-L1",
    "note.beamedLeft1-L4",
    "note.beamedRight1-L5",
    "dot-S1",
    "note.half-L2",
    "note.half-L5",
    "note.beamedLeft1-S3",
    "rest.sixteenth-L3",
    "note.beamedBoth2-S4",
    "note.beamedRight1-S2",
    "note.beamedLeft1-S4",
    "note.beamedLeft1-L3",
    "note.beamedLeft1-S2",
    "note.beamedLeft2-S4",
    "note.beamedLeft2-L3",
    "digit.8-L2",
    "accidental.flat-S2",
    "note.beamedBoth2-L3",
    "accidental.flat-L2",
    "note.beamedBoth2-L5",
    "note.beamedBoth2-S3",
    "dot-S6",
    "note.eighth-L6",
    "note.beamedLeft2-L5",
    "note.quarter-S0",
    "note.beamedRight2-L4",
    "note.beamedRight1-S5",
    "note.beamedRight1-L2",
    "note.beamedRight2-S4",
    "note.beamedRight2-S3",
    "note.beamedLeft2-S2",
    "note.beamedBoth2-S2",
    "digit.2-L2",
    "accidental.flat-S3",
    "note.beamedLeft1-L5",
    "clef.F-L4",
    "note.quarter-S6",
    "note.beamedBoth2-S5",
    "note.sixteenth-S3",
    "note.beamedLeft2-L2",
    "digit.2-S5",
    "accidental.sharp-L4",
    "note.beamedBoth1-S3",
    "note.beamedLeft1-L2",
    "note.half-S5",
    "note.beamedRight2-L3",
    "note.beamedLeft2-S5",
    "note.beamedRight2-L5",
    "note.beamedBoth1-L4",
    "accidental.sharp-S5",
    "note.beamedBoth1-S4",
    "note.beamedRight1-L6",
    "note.eighth-L1",
    "accidental.sharp-L1",
    "note.half-S1",
    "note.beamedBoth2-L2",
    "note.beamedBoth1-L3",
    "note.beamedLeft1-S5",
    "note.beamedRight1-S1",
    "note.sixteenth-L4",
    "note.sixteenth-S4",
    "note.beamedRight2-S2",
    "note.beamedBoth1-S2",
    "digit.6-L4",
    "note.beamedRight2-S5",
    "note.beamedBoth1-L5",
    "accidental.sharp-L3",
    "note.beamedBoth2-L6",
    "note.sixteenth-L3",
    "note.eighth-S0",
    "note.beamedBoth1-L2",
    "note.beamedLeft1-S1",
    "accidental.sharp-S4",
    "dot-S0",
    "note.sixteenth-L5",
    "note.beamedLeft2-L6",
    "note.half-L6",
    "note.beamedBoth2-S1",
    "fermata.above-S6",
    "note.beamedRight2-L2",
    "note.eighth-S6",
    "note.beamedLeft1-L1",
    "note.half-L1",
    "note.beamedRight1-L1",
    "note.beamedLeft2-S1",
    "note.sixteenth-S2",
    "note.quarter-L0",
    "note.beamedBoth1-S1",
    "accidental.sharp-L2",
    "accidental.natural-L3",
    "accidental.sharp-S1",
    "note.beamedRight1-S0",
    "note.beamedBoth1-S5",
    "note.whole-S3",
    "clef.C-L3",
    "digit.3-S5",
    "note.beamedLeft1-L6",
    "note.whole-L3",
    "note.beamedRight2-L6",
    "note.beamedRight1-S6",
    "note.whole-L4",
    "note.sixteenth-L2",
    "note.beamedBoth2-L1",
    "clef.C-L4",
    "digit.4-S5",
    "note.half-S0",
    "note.beamedLeft2-L1",
    "note.beamedBoth1-S0",
    "note.beamedRight2-S1",
    "note.beamedBoth2-S6",
    "note.beamedBoth1-L1",
    "note.beamedLeft2-S6",
    "slur.start-L4",
    "slur.end-L4",
    "note.beamedBoth2-S0",
    "note.beamedLeft1-S6",
    "note.whole-S2",
    "slur.start-S4",
    "slur.end-S4",
    "note.quarter-L7",
    "gracenote.eighth-S4",
    "note.beamedRight2-L1",
    "accidental.natural-L4",
    "note.sixteenth-S5",
    "accidental.natural-S3",
    "accidental.natural-S4",
    "gracenote.eighth-L4",
    "gracenote.eighth-S3",
    "note.beamedBoth3-L4",
    "note.whole-L2",
    "slur.start-L5",
    "slur.end-L5",
    "note.beamedBoth1-L6",
    "note.beamedBoth3-S3",
    "accidental.flat-S5",
    "note.quarter-S7",
    "note.half-S6",
    "note.beamedLeft3-S3",
    "digit.6-S5",
    "note.whole-S4",
    "gracenote.eighth-L5",
    "note.beamedLeft3-L4",
    "slur.start-S3",
    "slur.end-S3",
    "accidental.flat-L5",
    "note.beamedLeft2-S0",
    "note.beamedBoth3-L3",
    "digit.7-S5",
    "digit.5-S5",
    "note.beamedBoth3-S4",
    "note.beamedRight2-S6",
    "note.whole-L5",
    "digit.8-S5",
    "note.beamedLeft3-L3",
    "gracenote.eighth-S5",
    "note.beamedBoth2-L0",
    "note.beamedLeft1-S0",
    "note.beamedBoth3-S2",
    "gracenote.eighth-L3",
    "note.sixteenth-S1",
    "note.beamedRight2-S0",
    "note.beamedBoth1-S6",
    "note.beamedBoth3-L5",
    "note.eighth-L0",
    "accidental.natural-S2",
    "slur.start-L3",
    "slur.end-L3",
    "digit.0-S5",
    "note.quarter-S-1",
    "rest.whole-L4",
    "note.sixteenth-L6",
    "note.beamedRight1-L0",
    "note.beamedRight1-L7",
    "note.beamedLeft3-L5",
    "dot-S7",
    "slur.start-S5",
    "slur.end-S5",
    "note.whole-S1",
    "digit.4-L4",
    "digit.9-S5",
    "accidental.natural-S5",
    "gracenote.eighth-S2",
    "note.beamedLeft3-S4",
    "note.beamedBoth2-L7",
    "note.beamedRight3-L4",
    "note.sixteenth-L1",
    "note.beamedBoth3-S5",
    "note.beamedLeft2-L0",
    "note.half-L0",
    "slur.start-S2",
    "slur.end-S2",
    "note.beamedLeft2-L7",
    "note.beamedLeft3-S2",
    "note.beamedRight3-S2",
    "accidental.natural-L5",
    "accidental.natural-L2",
    "slur.start-L2",
    "slur.end-L2",
    "accidental.flat-S1",
    "note.beamedRight3-S3",
    "note.beamedLeft1-L0",
    "note.beamedBoth1-L0",
    "note.quarter-L-1",
    "dot-S-1",
    "note.beamedRight3-L3",
    "note.eighth-L7",
    "note.beamedLeft3-L2",
    "note.eighth-S7",
    "note.beamedRight1-S7",
    "note.beamedBoth3-L2",
    "clef.G-L1",
    "note.beamedBoth3-L6",
    "note.beamedLeft2-S-1",
    "note.beamedRight2-L0",
    "note.beamedLeft3-S5",
    "gracenote.sixteenth-S3",
    "accidental.sharp-L6",
    "note.whole-S5",
    "note.beamedLeft1-S-1",
    "note.beamedRight3-L5",
    "note.beamedRight2-L7",
    "note.beamedBoth3-L1",
    "note.whole-L1",
    "note.beamedLeft1-L7",
    "gracenote.sixteenth-L4",
    "gracenote.sixteenth-S4",
    "digit.12-L4",
    "note.eighth-S-1",
    "gracenote.eighth-L6",
    "slur.start-L6",
    "slur.end-L6",
    "gracenote.sixteenth-L5",
    "note.beamedRight1-S-1",
    "note.beamedLeft3-L1",
    "note.whole-S0",
    "note.beamedLeft3-L6",
    "note.beamedBoth1-L-1",
    "note.sixteenth-S0",
    "note.beamedRight3-S1",
    "note.beamedBoth2-S7",
    "accidental.natural-S1",
    "gracenote.sixteenth-L3",
    "note.beamedLeft3-S1",
    "rest.thirty_second-L3",
    "note.beamedBoth3-S1",
    "note.beamedBoth1-L7",
    "note.beamedRight3-S4",
    "note.beamedBoth3-S6",
    "note.beamedLeft1-L-1",
    "slur.start-S6",
    "slur.end-S6",
    "gracenote.eighth-L2",
    "note.beamedRight2-S7",
    "note.beamedBoth1-S-1",
    "note.beamedRight3-L2",
    "accidental.flat-L1",
    "note.half-S7",
    "note.sixteenth-S6",
    "gracenote.eighth-S1",
    "accidental.sharp-S0",
    "note.beamedBoth2-S-1",
    "accidental.natural-L1",
    "accidental.sharp-L0",
    "gracenote.sixteenth-S2",
    "accidental.flat-L6",
    "note.beamedRight1-L-1",
    "note.beamedLeft3-L0",
    "slur.start-S1",
    "slur.end-S1",
    "gracenote.sixteenth-L2",
    "note.half-L7",
    "note.beamedLeft1-S7",
    "note.eighth-L-1",
    "note.beamedLeft3-S6",
    "accidental.natural-S-1",
    "slur.start-S0",
    "slur.end-S0",
    "gracenote.sixteenth-S5",
    "note.quarter-S-2",
    "note.beamedLeft3-S0",
    "note.beamedBoth3-S0",
    "note.beamedLeft1-S-2",
    "clef.C-L2",
    "slur.start-L1",
    "slur.end-L1",
    "note.half-S-1",
    "accidental.natural-L6",
    "note.whole-L0",
    "note.beamedBoth1-S-2",
    "note.beamedBoth3-S-1",
    "digit.9-L4",
    "accidental.flat-S6",
    "note.beamedLeft2-S7",
    "accidental.flat-S-1",
    "note.beamedBoth3-L7",
    "slur.start-L7",
    "slur.end-L7",
    "note.beamedRight3-S6",
    "digit.1-L2",
    "accidental.natural-S6",
    "note.sixteenth-S7",
    "note.beamedRight2-S-1",
    "note.beamedLeft3-S-1",
    "note.beamedRight1-S-2",
    "note.beamedLeft3-L7",
    "note.thirty_second-S4",
    "note.beamedRight2-L-1",
    "note.whole-L7",
    "gracenote.eighth-L0",
    "note.whole-L6",
    "note.beamedBoth1-S7",
    "note.beamedBoth4-L3",
    "note.beamedRight3-S5",
    "note.sixteenth-L0",
    "gracenote.eighth-L1",
    "note.thirty_second-L2",
    "gracenote.thirty_second-L5",
    "note.beamedRight3-L0",
    "note.beamedLeft2-L-1",
    "note.beamedBoth3-L0",
    "note.double_whole-L2",
    "note.thirty_second-S3",
    "note.beamedBoth4-L5",
    "gracenote.sixteenth-S1",
    "note.beamedLeft2-S-2",
    "note.beamedRight3-L6",
    "gracenote.beamedBoth1-S4",
    "note.beamedRight3-S0",
    "note.beamedBoth4-L4",
    "note.beamedBoth4-S4",
    "note.thirty_second-L3",
    "note.double_whole-S3",
    "accidental.flat-S7",
    "note.beamedBoth3-L-1",
    "note.beamedLeft4-S5",
    "slur.start-S7",
    "slur.end-S7",
    "rest.sixty_fourth-L3",
    "gracenote.thirty_second-S5",
    "gracenote.thirty_second-L6",
    "note.eighth-L8",
    "note.beamedRight3-L1",
    "note.beamedBoth0-S4",
    "note.thirty_second-S2",
    "gracenote.sixteenth-L6",
    "note.thirty_second-L5",
    "accidental.natural-L0",
    "note.beamedBoth2-S-2",
    "note.beamedRight1-L8",
    "note.beamedLeft3-S7",
    "slur.start-L0",
    "slur.end-L0",
    "note.quarter-L8",
    "gracenote.beamedBoth1-L4",
    "note.beamedBoth4-S5",
    "gracenote.beamedBoth2-L4",
    "gracenote.beamedBoth1-S5",
    "note.beamedRight3-S7",
    "note.beamedBoth4-S3",
    "gracenote.thirty_second-S4",
    "note.half-L-1",
    "digit.16-L2",
    "note.eighth-S-2",
    "gracenote.thirty_second-L3",
    "note.whole-S6",
    "note.beamedBoth2-L-1",
    "note.eighth-S8",
    "gracenote.quarter-L4",
    "note.thirty_second-S5",
    "slur.start-S-1",
    "slur.end-S-1",
    "gracenote.beamedBoth2-S4",
    "note.beamedRight3-L7",
    "gracenote.sixteenth-L1",
    "note.beamedLeft0-L4",
    "gracenote.thirty_second-L4",
    "gracenote.quarter-S5",
    "accidental.sharp-L7",
    "note.double_whole-S4",
    "note.beamedRight0-S6",
    "note.beamedBoth2-L8",
    "note.thirty_second-L4",
    "accidental.natural-S0",
    "gracenote.quarter-L3",
    "gracenote.quarter-S2",
    "note.beamedRight0-S5",
    "clef.F-L3",
    "note.beamedLeft0-S5",
    "note.beamedBoth3-S7",
    "gracenote.eighth-S0",
    "gracenote.thirty_second-S0",
    "note.beamedLeft0-S3",
    "note.beamedRight2-S-2",
    "note.beamedLeft1-L8",
    "dot-S8",
    "note.beamedBoth4-S1",
    "note.beamedBoth4-L2",
    "note.beamedBoth4-S2",
    "note.beamedLeft4-S3",
    "note.beamedRight4-S2",
    "note.beamedRight2-L8",
    "accidental.flat-S0",
    "accidental.sharp-S6",
    "note.beamedRight0-L3",
    "note.beamedBoth0-L3",
    "note.half-L8",
    "note.beamedBoth0-S3",
    "accidental.natural-L7",
    "note.sixteenth-L7",
    "accidental.sharp-S-2",
    "note.double_whole-L3",
    "note.beamedLeft4-L2",
    "note.beamedRight0-S3",
    "gracenote.beamedBoth1-S3",
    "note.beamedBoth0-L4",
    "slur.start-L-1",
    "slur.end-L-1",
    "gracenote.beamedBoth2-S3",
    "note.beamedLeft4-L5",
    "note.beamedLeft4-L6",
    "note.beamedLeft4-S6",
    "gracenote.beamedBoth2-L5",
    "note.beamedBoth4-L1",
    "note.beamedLeft4-S2",
    "note.thirty_second-L1",
    "note.beamedBoth0-L2",
    "gracenote.beamedBoth1-L3",
    "gracenote.quarter-S4",
    "note.quarter-L-3",
    "note.double_whole-S2",
    "gracenote.beamedBoth2-S2",
    "note.sixteenth-L-1",
    "note.beamedLeft4-S4",
    "gracenote.thirty_second-S3",
    "dot-S-2",
    "gracenote.sixteenth-S0",
    "gracenote.thirty_second-L2",
    "note.thirty_second-L-1",
    "note.whole-S-1",
    "note.thirty_second-L6",
    "note.beamedLeft0-L5",
    "note.thirty_second-S6",
    "accidental.flat-L7",
    "gracenote.beamedBoth2-S5",
    "note.beamedRight3-S-2",
    "note.beamedLeft0-S2",
    "gracenote.beamedBoth1-S2",
    "gracenote.sixteenth-L-1",
    "note.thirty_second-S1",
    "note.beamedRight0-S2",
    "gracenote.beamedBoth1-L5",
    "gracenote.quarter-S3",
    "gracenote.beamedBoth3-L6",
    "note.beamedBoth0-S2",
    "note.beamedRight0-L5",
    "note.double_whole-S6",
    "note.eighth-L-2",
]

BLANK_IDX  = 0
VOCAB_SIZE = len(VOCAB)
SYM_TO_IDX = {s: i for i, s in enumerate(VOCAB)}
IDX_TO_SYM = {i: s for i, s in enumerate(VOCAB)}

# ── Model ─────────────────────────────────────────────────────────────────────
class ResNetBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class CRNN(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, rnn_hidden=256, rnn_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            ResNetBlock(64),
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            ResNetBlock(128), ResNetBlock(128),
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),
            ResNetBlock(256), ResNetBlock(256),
            nn.Conv2d(256, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.AdaptiveAvgPool2d((1, None)),
        )
        self.rnn = nn.LSTM(
            input_size=512, hidden_size=rnn_hidden,
            num_layers=rnn_layers, batch_first=False,
            bidirectional=True,
            dropout=0.3 if rnn_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(rnn_hidden * 2, vocab_size)

    def forward(self, x):
        features = self.cnn(x)
        features = features.squeeze(2)
        features = features.permute(2, 0, 1)
        output, _ = self.rnn(features)
        output = self.fc(output)
        return F.log_softmax(output, dim=2)


# ── Greedy CTC decode ─────────────────────────────────────────────────────────
def _greedy_ctc_decode(log_probs):
    indices = torch.argmax(log_probs, dim=1).tolist()
    decoded, prev = [], None
    for idx in indices:
        if idx != BLANK_IDX and idx != prev:
            decoded.append(idx)
        prev = idx
    return decoded


# ── Symbol Error Rate ─────────────────────────────────────────────────────────
def compute_ser(predicted, ground_truth):
    n = len(ground_truth)
    if n == 0:
        return 0.0
    m = len(predicted)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): dp[i][0] = i
    for j in range(m + 1): dp[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ground_truth[i-1] == predicted[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[n][m] / n

print ("Done Initializing")

In [ ]:
# Diagnostic (exploratory, NOT part of the PIYA-106/BUG-011 font-retraining plan) --
# checks whether the *original* PrIMuS corpus (umerellous/primus/primus -- a
# separate Kaggle dataset upload, NOT the piamac/cameraprimus dataset VOCAB
# above was built from) contains .agnostic tokens absent from the current
# hardcoded VOCAB. Relocated here (was previously placed after cell-8's
# train() call, so it could never run without a full 50-epoch training pass
# first, and had thrown NameError: VOCAB from being executed out of order).
#
# CAVEATS before trusting this cell's output:
#   - The /kaggle/input/datasets/umerellous/primus/primus path is unverified --
#     confirm that dataset is actually attached to this notebook before relying
#     on a non-empty (or empty) result.
#   - "New tokens" here conflates two different things: tokens truly absent
#     from CameraPrIMuS vs. tokens that exist there but were dropped from VOCAB
#     by cell-4's min_count=2 frequency filter. A nonzero count below does NOT
#     by itself mean VOCAB needs regenerating -- check which case applies
#     before drawing that conclusion.
#   - If original PrIMuS and CameraPrIMuS share underlying musical content
#     (CameraPrIMuS is commonly described as PrIMuS + synthetic camera
#     distortion over the same incipits), any large token-coverage gap found
#     here would itself be surprising and worth root-causing before acting on
#     it, rather than assumed to reflect a real corpus difference.
#   - Regenerating VOCAB from a union of both corpora would change every
#     token's index (SYM_TO_IDX/IDX_TO_SYM), invalidating the hardcoded VOCAB
#     in cell-5 above and any checkpoint already trained against it. Do not
#     regenerate VOCAB off this cell's output without a fresh ml-cv-lead
#     review and an explicit decision doc -- this is scope creep relative to
#     BUG-011 (a font/glyph rendering gap, not a token-vocabulary gap) and
#     should not block Step 1/Step 2 of the font-retraining plan.

import collections
from pathlib import Path

current_vocab = set(VOCAB)
new_tokens = collections.Counter()

for f in Path("/kaggle/input/datasets/umerellous/primus/primus").rglob("*.agnostic"):
    with open(f) as file:
        for token in file.read().split():
            if token not in current_vocab:
                new_tokens[token] += 1

print(f"New tokens not in current VOCAB: {len(new_tokens)}")
for token, count in new_tokens.most_common(20):
    print(f"  {token}: {count}")

In [ ]:
import os, glob, torch
from torch.utils.data import Dataset
import cv2, numpy as np
from torchvision import transforms

class CameraPrIMuSDataset(Dataset):
    def __init__(self, root_dir: str, vocab: dict,
                 img_height: int = 64, augment: bool = False,
                 distorted_prob: float = 0.5, force_distorted: bool = False):
        """distorted_prob: when augment=True, fraction of samples drawn from the
        _distorted.jpg camera-simulation variant instead of the clean .png render
        (falls back to clean if a sample has no distorted file on disk). Session
        measurement (2026-09-09): 0.70% SER on clean vs 18.76% SER on
        _distorted.jpg for a checkpoint that never trained on the distorted
        variant at all -- this is the primary lever for closing that gap.
        force_distorted: always use the distorted variant when it exists, with
        no random draw and no extra synthetic _augment() on top -- for a
        dedicated "distorted-only" validation pass, independent of augment.
        """
        self.vocab          = vocab          # SYM_TO_IDX from your stage3 file
        self.img_height      = img_height
        self.augment          = augment
        self.distorted_prob  = distorted_prob
        self.force_distorted = force_distorted
        self.samples          = []  # (clean_img_path, distorted_img_path_or_None, label_list)

        # Gather all sample folders. sorted() makes iteration order deterministic across
        # Kaggle sessions/dataset mounts -- glob.glob's raw order depends on filesystem
        # directory-entry enumeration, which is not a guaranteed-stable contract even for
        # "the same" directory across separate mounts. Root-caused as the reason a
        # positional train/val split (see train() below) couldn't be exactly reproduced
        # in a later session (docs/bug-tracker.md BUG-011 notes, ml-cv-lead review
        # 2026-09-15). Doesn't change which samples exist, only the index<->sample mapping
        # any positional split relies on.
        for agnostic_path in sorted(glob.glob(
                f"{root_dir}/**/*.agnostic", recursive=True)):
            img_path = agnostic_path.replace(".agnostic", ".png")
            if not os.path.exists(img_path):
                continue
            distorted_path = agnostic_path.replace(".agnostic", "_distorted.jpg")
            if not os.path.exists(distorted_path):
                distorted_path = None
            with open(agnostic_path) as f:
                raw_tokens = f.read().split()
            label = [vocab.get(t, vocab["<unk>"]) for t in raw_tokens]
            if len(label) > 0:
                self.samples.append((img_path, distorted_path, label))

        n_distorted = sum(1 for _, d, _ in self.samples if d is not None)
        print(f"Loaded {len(self.samples)} samples from {root_dir} "
              f"({n_distorted} have a _distorted.jpg camera-simulation variant)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, distorted_path, label = self.samples[idx]

        if self.force_distorted and distorted_path is not None:
            use_distorted = True
        elif self.augment and distorted_path is not None:
            use_distorted = np.random.rand() < self.distorted_prob
        else:
            use_distorted = False

        src_path = distorted_path if use_distorted else img_path
        img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            # _distorted.jpg occasionally fails to decode -- fall back to the clean render
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise RuntimeError(f"Could not read image: {img_path}")
        h, w = img.shape
        new_w = max(1, int(w * self.img_height / h))
        img = cv2.resize(img, (new_w, self.img_height),
                         interpolation=cv2.INTER_AREA)

        if self.augment:
            img = self._augment(img)

        img = img.astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5
        tensor = torch.tensor(img).unsqueeze(0)   # (1, H, W)

        return tensor, torch.tensor(label, dtype=torch.long)

    def _augment(self, img: np.ndarray) -> np.ndarray:
        # Brightness/contrast jitter for camera robustness. Widened from the
        # original [0.8,1.2]/[-20,20] -- Camera-PrIMuS's own distortion pipeline
        # (Calvo-Zaragoza & Rizo, ISMIR 2018) lists "irregular lighting" as one
        # of its three named distortion categories, and the narrower range
        # under-covers the brightness/contrast swings phone auto-exposure
        # produces on a real photographed page.
        alpha = np.random.uniform(0.6, 1.4)  # contrast
        beta  = np.random.randint(-35, 35)   # brightness
        img   = np.clip(alpha * img + beta, 0, 255).astype(np.uint8)

        # Directional shadow / uneven-lighting gradient -- a uniform global
        # brightness/contrast jitter can't represent one side of a handheld
        # photo being darker than the other. Fills the gap between the paper's
        # "irregular lighting" category and the OCR/HTR "shadow/background
        # disturbance" technique, neither of which the uniform jitter above
        # covers on its own.
        if np.random.rand() < 0.3:
            img = self._add_shadow(img)

        # Small random rotation -- previously missing entirely. Camera-PrIMuS's
        # distortion pipeline explicitly names rotation as one of its three
        # distortion categories; horizontal stretch and perspective warp below
        # are different transforms and don't substitute for it. Kept to a small
        # +/-3 deg range since strips are already tightly cropped to one
        # staff's height (STRIP_HEIGHT=64 in stage2) -- a larger angle risks
        # rotating stafflines/noteheads out of frame.
        if np.random.rand() < 0.3:
            angle = np.random.uniform(-3.0, 3.0)
            img = self._rotate(img, angle)

        # Small random horizontal stretch (camera distortion sim)
        if np.random.rand() < 0.3:
            scale = np.random.uniform(0.9, 1.1)
            new_w = max(1, int(img.shape[1] * scale))
            img = cv2.resize(img, (new_w, img.shape[0]))

        # Blur -- defocus or handheld motion, camera photos are rarely
        # pixel-sharp. Kernel range widened to include 7 (previously capped at
        # 5): handheld motion blur is often stronger than mild defocus alone.
        if np.random.rand() < 0.35:
            k = int(np.random.choice([3, 5, 7]))
            if np.random.rand() < 0.5:
                img = cv2.GaussianBlur(img, (k, k), 0)
            else:
                kernel = np.zeros((k, k), dtype=np.float32)
                kernel[k // 2, :] = 1.0 / k
                img = cv2.filter2D(img, -1, kernel)

        # Random perspective warp -- page rarely perfectly frontal to camera
        if np.random.rand() < 0.3:
            img = self._random_perspective(img)

        # Elastic distortion -- paper curl / lens distortion. OCR/HTR
        # robustness literature reports this as the single highest-impact
        # augmentation for closing a clean/distorted accuracy gap, so
        # probability raised 0.2->0.35 here (intensity left unchanged --
        # alpha/sigma are already tuned to stay legible on thin stafflines;
        # going stronger risks destroying notehead shapes the CTC head
        # depends on).
        if np.random.rand() < 0.35:
            img = self._elastic_distort(img)

        return img

    def _rotate(self, img: np.ndarray, angle_deg: float) -> np.ndarray:
        h, w = img.shape
        matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle_deg, 1.0)
        return cv2.warpAffine(img, matrix, (w, h), borderValue=255,
                               flags=cv2.INTER_LINEAR)

    def _add_shadow(self, img: np.ndarray) -> np.ndarray:
        """Simulates uneven lighting with a linear darkening gradient across
        the strip, in a random direction and orientation -- approximates the
        shadow a handheld phone throws across part of a page, which the
        uniform brightness/contrast jitter above can't represent on its own.
        """
        h, w = img.shape
        if np.random.rand() < 0.5:
            ramp = np.linspace(0, 1, w, dtype=np.float32)
            if np.random.rand() < 0.5:
                ramp = ramp[::-1]
            gradient = np.tile(ramp, (h, 1))
        else:
            ramp = np.linspace(0, 1, h, dtype=np.float32)
            if np.random.rand() < 0.5:
                ramp = ramp[::-1]
            gradient = np.tile(ramp.reshape(-1, 1), (1, w))
        strength = np.random.uniform(0.15, 0.45)  # max darkening fraction
        darken = 1.0 - gradient * strength
        shadowed = img.astype(np.float32) * darken
        return np.clip(shadowed, 0, 255).astype(np.uint8)

    def _random_perspective(self, img: np.ndarray) -> np.ndarray:
        h, w = img.shape
        jitter = 0.04  # max corner displacement, as a fraction of each dimension
        src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
        offsets = np.random.uniform(-jitter, jitter, src.shape) * np.array([w, h])
        dst = (src + offsets).astype(np.float32)
        matrix = cv2.getPerspectiveTransform(src, dst)
        return cv2.warpPerspective(img, matrix, (w, h), borderValue=255)

    def _elastic_distort(self, img: np.ndarray, alpha: float = 6.0, sigma: float = 4.0) -> np.ndarray:
        h, w = img.shape
        dx = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigma) * alpha
        dy = cv2.GaussianBlur((np.random.rand(h, w).astype(np.float32) * 2 - 1), (0, 0), sigma) * alpha
        x, y = np.meshgrid(np.arange(w), np.arange(h))
        map_x = (x + dx).astype(np.float32)
        map_y = (y + dy).astype(np.float32)
        return cv2.remap(img, map_x, map_y, interpolation=cv2.INTER_LINEAR, borderValue=255)

In [ ]:
def ctc_collate(batch):
    """Pad images to same width; pack labels for CTC loss."""
    images, labels = zip(*batch)

    # Pad images to max width in batch
    max_w = max(img.shape[2] for img in images)
    padded = torch.zeros(len(images), 1, images[0].shape[1], max_w)
    for i, img in enumerate(images):
        padded[i, :, :, :img.shape[2]] = img

    # Input lengths = time steps after CNN (width // 4 due to pooling)
    # Your CNN applies 2 MaxPool2d(2,2) on width and 2 MaxPool2d((2,1))
    # So width is halved twice: T = W // 4
    input_lengths = torch.tensor(
        [img.shape[2] // 4 for img in images], dtype=torch.long)

    # Flatten labels and record lengths
    label_lengths = torch.tensor(
        [len(lb) for lb in labels], dtype=torch.long)
    flat_labels = torch.cat(labels)

    return padded, flat_labels, input_lengths, label_lengths

In [ ]:
import os, time
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

def _evaluate_ser(model, loader, device):
    """Mean SER over one validation loader. Its own function so the same
    eval logic can run twice per epoch (clean vs. distorted variant) without
    duplicating the CTC-decode/SER-alignment logic."""
    model.eval()
    all_ser = []
    with torch.no_grad():
        for images, labels, in_lens, lb_lens in loader:
            images    = images.to(device)
            log_probs = model(images)   # (T, B, V)

            label_offset = 0
            for b in range(images.size(0)):
                lp       = log_probs[:, b, :]
                pred_ids = _greedy_ctc_decode(lp)

                gt_len = lb_lens[b].item()
                gt_ids = labels[label_offset : label_offset + gt_len].tolist()
                label_offset += gt_len

                pred_syms = [IDX_TO_SYM.get(i, "<unk>") for i in pred_ids]
                gt_syms   = [IDX_TO_SYM.get(i, "<unk>") for i in gt_ids]
                all_ser.append(compute_ser(pred_syms, gt_syms))
    return float(np.mean(all_ser)) if all_ser else 0.0


def train(root_dir="/kaggle/input/datasets/piamac/cameraprimus/CameraPrIMuS - Copy",
          weights_out="crnn_weights/best.pt",
          img_height=64, epochs=50, batch_size=16, lr=3e-4,
          distorted_prob=0.5):
    """distorted_prob: fraction of augmented training samples drawn from the
    _distorted.jpg camera-simulation variant instead of the clean .png render
    -- see CameraPrIMuSDataset's docstring for the measurement that motivates
    this default.
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # -- Data -- separate dataset instances so augment/eval-mode doesn't bleed over.
    # All three instances glob the same root_dir in the same (deterministic) order,
    # so a given index refers to the same underlying sample across all of them --
    # the same assumption the original train/val split already relied on.
    full_train_ds         = CameraPrIMuSDataset(root_dir, SYM_TO_IDX, img_height=img_height,
                                                 augment=True, distorted_prob=distorted_prob)
    full_val_clean_ds     = CameraPrIMuSDataset(root_dir, SYM_TO_IDX, img_height=img_height,
                                                 augment=False)
    full_val_distorted_ds = CameraPrIMuSDataset(root_dir, SYM_TO_IDX, img_height=img_height,
                                                 augment=False, force_distorted=True)

    # Split indices -- same seed so train/val never overlap
    indices  = list(range(len(full_train_ds)))
    val_size = int(0.1 * len(indices))
    train_indices = indices[val_size:]   # 90%
    val_indices   = indices[:val_size]   # 10%

    train_ds         = Subset(full_train_ds,         train_indices)
    val_ds_clean      = Subset(full_val_clean_ds,      val_indices)
    val_ds_distorted  = Subset(full_val_distorted_ds,  val_indices)

    print(f"Train: {len(train_ds):,} samples | "
          f"Val: {len(val_ds_clean):,} samples (evaluated clean AND distorted)")

    train_loader        = DataLoader(train_ds, batch_size=batch_size,
                                     shuffle=True,  collate_fn=ctc_collate,
                                     num_workers=0, pin_memory=True)
    val_loader_clean     = DataLoader(val_ds_clean, batch_size=batch_size,
                                     shuffle=False, collate_fn=ctc_collate,
                                     num_workers=0, pin_memory=True)
    val_loader_distorted = DataLoader(val_ds_distorted, batch_size=batch_size,
                                     shuffle=False, collate_fn=ctc_collate,
                                     num_workers=0, pin_memory=True)

    # -- Model
    model     = CRNN(VOCAB_SIZE).to(device)
    criterion = torch.nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # "Best" checkpoint is selected on DISTORTED SER, not clean -- that's the
    # metric that matches real (camera-photo) deployment. clean_ser is tracked
    # and printed alongside it purely as a canary: if it starts climbing while
    # distorted_ser falls, fine-tuning is trading away clean-image accuracy for
    # distortion robustness (the failure mode documented in arXiv 2202.10054),
    # and that trade-off should be a deliberate call, not something silent.
    best_distorted_ser = float("inf")
    start_epoch = 1
    os.makedirs("crnn_weights", exist_ok=True)

    # -- Resume from checkpoint if it exists --------------------------------
    if os.path.exists(weights_out):
        checkpoint = torch.load(weights_out, map_location=device)
        model.load_state_dict(checkpoint["model_state"])
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
        best_distorted_ser = checkpoint.get("distorted_ser", checkpoint.get("ser", float("inf")))
        start_epoch = checkpoint["epoch"] + 1
        print(f"  ⇺ Resumed from epoch {checkpoint['epoch']} "
              f"(best distorted SER={best_distorted_ser:.4f})")

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()

        # -- Train loop
        model.train()
        train_loss = 0.0
        for images, labels, in_lens, lb_lens in train_loader:
            images  = images.to(device)
            labels  = labels.to(device)
            in_lens = in_lens.to(device)
            lb_lens = lb_lens.to(device)

            log_probs = model(images)                          # (T, B, V)
            in_lens   = in_lens.clamp(max=log_probs.shape[0]) # safety clamp

            loss = criterion(log_probs, labels, in_lens, lb_lens)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()

        # -- Validation -- clean and distorted tracked separately (see docstring above)
        clean_ser     = _evaluate_ser(model, val_loader_clean, device)
        distorted_ser = _evaluate_ser(model, val_loader_distorted, device)
        epoch_time    = time.time() - t0

        print(f"Epoch {epoch:03d} | "
              f"loss {train_loss / len(train_loader):.4f} | "
              f"clean SER {clean_ser:.4f} | "
              f"distorted SER {distorted_ser:.4f} | "
              f"{epoch_time:.1f}s")

        if distorted_ser < best_distorted_ser:
            best_distorted_ser = distorted_ser
            torch.save({
                "model_state":     model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "epoch":           epoch,
                "distorted_ser":   best_distorted_ser,
                "clean_ser":       clean_ser,
                "ser":             best_distorted_ser,  # backward-compat key for any external reader expecting "ser"
                "vocab":           VOCAB,
            }, weights_out)
            print(f"  ✓ Saved best model (distorted SER={best_distorted_ser:.4f}, "
                  f"clean SER={clean_ser:.4f}) → {weights_out}")

    print(f"\nTraining complete. Best distorted SER: {best_distorted_ser:.4f}")

RUN_BASE_TRAINING = False  # base CameraPrIMuS run already completed and checkpointed
# (persisted as the piamac/crnn-cameraprimus-transcription-model Kaggle Model). This
# guard stops Save & Run All from re-triggering a full 50-epoch run from scratch
# before this notebook ever reaches step 2 below. Flip to True and re-run this cell
# only if the base run itself needs to be redone.
if RUN_BASE_TRAINING:
    train()

# Step 2 — Fine-Tuning on Font-Augmented Data (PIYA-106 BUG-011)

Implements step 2 of `docs/decisions/piya-106-crnn-font-retraining-plan.md`, per the
2026-09-15 ml-cv-lead review of this specific plan. Fine-tunes the already-trained base
checkpoint (`piamac/crnn-cameraprimus-transcription-model`) on the step-1 font-augmented
renders (`piamac/cpm-font-augmented`: Bravura/Gootville/Leland/Petaluma, 0 failures /
15,980 renders) so the CRNN stops collapsing to `clef.C-L3` on real, non-CPM-font
screenshots (BUG-011).

**Locked-in requirements from the ml-cv-lead review** (all implemented below):
1. **Fine-tune from the current base checkpoint**, not a fresh base run (explicit user
   decision, 2026-09-15).
2. **Stem-based train/val split**, not row-based — every (clean, distorted, Bravura,
   Gootville, Leland, Petaluma) variant of one incipit stays entirely in train or
   entirely in val, so the model can't "cheat" by recognizing memorized melodic content
   across a font it's supposedly generalizing to.
3. **Forgetting canary reuses the EXACT held-out set** that produced the recorded
   ~98%/87% clean/distorted baseline SER (`docs/bug-tracker.md`, BUG-011) — not a fresh
   random slice, which wouldn't be comparable.
4. **Canary floor derived from that set's own statistical noise** (bootstrap 95% CI upper
   bound on SER), not a round number picked in advance.
5. **~~Joint checkpoint-selection criterion~~ — AMENDED 2026-09-15** (see
   `docs/decisions/piya-106-crnn-font-retraining-plan.md`'s amendment section): the first
   LR sweep breached the bootstrap-CI floor at epoch 1 for every candidate LR, before any
   checkpoint could be saved. Root cause: that CI bounds *resampling* variance, not
   *weight-perturbation* variance, so it was expected to trip on essentially any real
   gradient update. The floor is now a **diagnostic-only** signal (still logged every
   epoch); the sole enforced auto-stop is a looser **hard cap** (an explicit,
   human-picked multiplier on each canary subset's own baseline mean). Every epoch now
   saves a checkpoint unconditionally, tagged with the gate mode and thresholds in
   effect, so a full trend is always recoverable regardless of which tier fired.
6. **LR chosen by a short sweep against this data**, not picked a priori.

**Scope caveat (from the same review, carried forward, not yet addressed):** this step
tests font-domain generalization in isolation (bare re-renders, Condition A only, no
extra distortion pass beyond what step-1 already inherits from `CameraPrIMuSDataset`
reusing `_augment()`). It does **not** test the compound font+camera-distortion condition
BUG-011 was actually filed against (real phone screenshots) — that's a separate,
still-unscheduled evaluation step (steps 3–7 of the decision doc) and should not be
conflated with what this step's numbers measure.

**font_val_ser leakage caveat (flagged in Cell B, not yet fixed):** the font-val split's
stems overlap heavily with the base model's original CameraPrIMuS training stems, so a
low font_val_ser partly reflects re-exposure to already-memorized content in a new font,
not pure font-domain generalization. Do not cite font_val_ser as generalization evidence
in a go/no-go call until a disjoint held-out font-val set exists (tracked as a separate
follow-up, not part of this amendment).

**Cells below are guarded** (`RUN_...` flags default `False`) so they define functions
and inspect data without spending compute or accidentally training on a full-notebook
Save & Run All — flip a flag and re-run only that cell when actually ready to execute it.

In [ ]:
# Step 2, Cell A — forgetting canary: reproduce the EXACT held-out set that produced
# the recorded ~98%/87% clean/distorted baseline (docs/bug-tracker.md, BUG-011), load
# the base checkpoint, and derive a canary floor from that set's own bootstrap CI --
# not a round number, per the ml-cv-lead review.

BASE_CHECKPOINT_PATH = ("/kaggle/input/models/piamac/crnn-cameraprimus-transcription-model"
                         "/pytorch/base-checkpoint/1/crnn_cameraprimus_transcription.pt")
CANARY_ROOT = "/kaggle/input/datasets/piamac/cameraprimus/CameraPrIMuS - Copy"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Same construction as train()'s split above: CameraPrIMuSDataset globs root_dir in a
# fixed (unsorted) order; indices[:val_size] is the exact 10% slice used as the base
# run's val set. Reproducing it here -- not a fresh random slice -- is required for the
# canary numbers below to be comparable to the recorded baseline.
_canary_clean_full     = CameraPrIMuSDataset(CANARY_ROOT, SYM_TO_IDX, img_height=64, augment=False)
_canary_distorted_full = CameraPrIMuSDataset(CANARY_ROOT, SYM_TO_IDX, img_height=64,
                                              augment=False, force_distorted=True)

_canary_indices  = list(range(len(_canary_clean_full)))
_canary_val_size = int(0.1 * len(_canary_indices))
CANARY_VAL_INDICES = _canary_indices[:_canary_val_size]         # == train()'s val_indices
BASE_TRAIN_INDICES = set(_canary_indices[_canary_val_size:])    # == train()'s train_indices

canary_clean_loader = DataLoader(Subset(_canary_clean_full, CANARY_VAL_INDICES),
                                  batch_size=16, shuffle=False, collate_fn=ctc_collate)
canary_distorted_loader = DataLoader(Subset(_canary_distorted_full, CANARY_VAL_INDICES),
                                      batch_size=16, shuffle=False, collate_fn=ctc_collate)

print(f"Canary set: {len(CANARY_VAL_INDICES)} samples (same held-out slice as the base training run)")

# -- Load the base checkpoint --------------------------------------------------------
model = CRNN(VOCAB_SIZE).to(device)
_base_ckpt = torch.load(BASE_CHECKPOINT_PATH, map_location=device)
assert _base_ckpt.get("vocab") == VOCAB, (
    "Checkpoint's stored VOCAB doesn't match this notebook's hardcoded VOCAB (cell-5) -- "
    "token indices would silently misalign if loaded anyway.")
model.load_state_dict(_base_ckpt["model_state"])
print(f"Loaded base checkpoint from epoch {_base_ckpt.get('epoch')} "
      f"(recorded distorted SER={_base_ckpt.get('distorted_ser', _base_ckpt.get('ser')):.4f}, "
      f"clean SER={_base_ckpt.get('clean_ser', float('nan')):.4f})")


def _per_sample_ser(model, loader, device):
    """Like _evaluate_ser above, but returns every sample's SER instead of just the
    mean -- needed to bootstrap a CI rather than report a single number."""
    model.eval()
    sers = []
    with torch.no_grad():
        for images, labels, in_lens, lb_lens in loader:
            images = images.to(device)
            log_probs = model(images)
            label_offset = 0
            for b in range(images.size(0)):
                lp = log_probs[:, b, :]
                pred_ids = _greedy_ctc_decode(lp)
                gt_len = lb_lens[b].item()
                gt_ids = labels[label_offset:label_offset + gt_len].tolist()
                label_offset += gt_len
                pred_syms = [IDX_TO_SYM.get(i, "<unk>") for i in pred_ids]
                gt_syms = [IDX_TO_SYM.get(i, "<unk>") for i in gt_ids]
                sers.append(compute_ser(pred_syms, gt_syms))
    return sers


baseline_clean_sers     = _per_sample_ser(model, canary_clean_loader, device)
baseline_distorted_sers = _per_sample_ser(model, canary_distorted_loader, device)

print(f"Canary reproduction check -- clean SER: {np.mean(baseline_clean_sers):.4f} "
      f"(historical reference ~0.02, i.e. ~98% per docs/bug-tracker.md)")
print(f"Canary reproduction check -- distorted SER: {np.mean(baseline_distorted_sers):.4f} "
      f"(historical reference ~0.13, i.e. ~87% per docs/bug-tracker.md)")
print(
    "NOTE (ml-cv-lead review, 2026-09-15): a mismatch here is EXPECTED, not a live bug -- "
    "confirmed by two independent root causes: (1) CameraPrIMuSDataset's glob order (now "
    "sorted() in cell-7 going forward) was never guaranteed stable across the Kaggle "
    "session that produced this checkpoint vs. this one, so a positional 90/10 split "
    "cannot be exactly re-derived after the fact; (2) this checkpoint predates clean_ser "
    "tracking entirely (clean SER printed as nan above), so it may not even share the "
    "current split logic. There is no artifact from the original run (no persisted path "
    "list/seed) to recover the historical split from -- this is a dead end, not something "
    "to keep debugging. The historical ~98%/87% figures are also a different metric "
    "(pitch accuracy over a separate 500-sample full-pipeline eval) than the SER measured "
    "here, so exact agreement was never guaranteed even with a perfectly reproduced split. "
    "The floor below is therefore SESSION-LOCAL: an honest measurement of this "
    "checkpoint's performance on this held-out slice, used only to detect regression "
    "during fine-tuning within this run -- not a reproduction of the historical baseline.")


def bootstrap_ci_upper(samples, n_boot=2000, ci=0.95, seed=0):
    """Upper bound of a 95% bootstrap CI on the mean SER -- used as the canary floor:
    an SER above this is distinguishable from the baseline checkpoint's own re-sampling
    noise on this exact set, not just noise, at the given confidence level."""
    rng = np.random.default_rng(seed)
    arr = np.asarray(samples)
    means = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)]
    return float(np.percentile(means, ci * 100))


CANARY_CLEAN_FLOOR_SER     = bootstrap_ci_upper(baseline_clean_sers)
CANARY_DISTORTED_FLOOR_SER = bootstrap_ci_upper(baseline_distorted_sers)
print(f"\nSession-local canary floor (95% CI upper bound on THIS run's own measurement) -- "
      f"clean: {CANARY_CLEAN_FLOOR_SER:.4f}, distorted: {CANARY_DISTORTED_FLOOR_SER:.4f}")
_hist_clean, _hist_distorted = 0.02, 0.13
print(f"(historical reference {_hist_clean:.4f}/{_hist_distorted:.4f} falls outside this "
      f"CI on both -- expected per the note above, not evidence of a new problem)")

# -- Amendment 2026-09-15 (see docs/decisions/piya-106-crnn-font-retraining-plan.md) -------
# The bootstrap CI above bounds RESAMPLING variance (which 399 canary items got drawn), not
# WEIGHT-PERTURBATION variance (how much this eval set moves once real gradient steps are
# taken). The first LR sweep breached it at epoch 1 for all three candidates -- close to
# that gate's expected behavior, not proof fine-tuning is unsafe. It is kept below as a
# DIAGNOSTIC signal only (still logged every epoch). The sole enforced auto-stop is now a
# looser HARD CAP, an explicit human judgment call (picked for headroom above the worst
# epoch-1 sweep result, ~0.0955 clean SER at lr=1e-4) -- not statistically derived, same
# caveat as the floor above. Revisit this multiplier if the 3-epoch trend rerun (Cell C)
# shows it's miscalibrated in either direction.
CANARY_HARD_CAP_MULTIPLIER = 3.0  # judgment call -- see docs/decisions/... 2026-09-15 amendment
CANARY_CLEAN_HARD_CAP_SER     = float(np.mean(baseline_clean_sers))     * CANARY_HARD_CAP_MULTIPLIER
CANARY_DISTORTED_HARD_CAP_SER = float(np.mean(baseline_distorted_sers)) * CANARY_HARD_CAP_MULTIPLIER
print(f"\nCanary gate (two-tier, amended 2026-09-15):")
print(f"  soft/diagnostic (bootstrap CI, logged only, never stops a run) -- "
      f"clean: {CANARY_CLEAN_FLOOR_SER:.4f}, distorted: {CANARY_DISTORTED_FLOOR_SER:.4f}")
print(f"  hard/enforced (auto-stop, {CANARY_HARD_CAP_MULTIPLIER}x each subset's own baseline mean) -- "
      f"clean: {CANARY_CLEAN_HARD_CAP_SER:.4f}, distorted: {CANARY_DISTORTED_HARD_CAP_SER:.4f}")
print("Fine-tuning (Cell C below) only auto-stops on a HARD CAP breach; soft-floor "
      "breaches are logged every epoch but are not fatal.")

In [ ]:
# Step 2, Cell B — combined fine-tuning dataset: original CameraPrIMuS (clean +
# _augment()-distorted) plus all 4 font-augmented directories, split by STEM (not row)
# so the same incipit never appears in both train and val across different fonts.
# Canary stems are excluded entirely so the forgetting probe stays a clean held-out set.

from collections import defaultdict
import random

FONT_DATA_ROOT = Path("/kaggle/input/datasets/piamac/cpm-font-augmented2")
LOO_FONT = "Leland"  # held out of training entirely -- MuseScore 4 default font, see docstring above
FONTS = ["Bravura", "Gootville", "Petaluma"]  # Leland excluded -- LOO probe


def _collect_samples_by_stem(root_dir):
    """dict: stem -> list of (img_path, distorted_path_or_None, label) -- same 3-tuple
    shape CameraPrIMuSDataset.samples uses, so CombinedFineTuneDataset below can reuse
    its __getitem__/_augment pipeline unchanged."""
    by_stem = defaultdict(list)
    for agnostic_path in Path(root_dir).rglob("*.agnostic"):
        img_path = agnostic_path.with_suffix(".png")
        if not img_path.exists():
            continue
        distorted_path = agnostic_path.with_name(agnostic_path.stem + "_distorted.jpg")
        distorted_path = str(distorted_path) if distorted_path.exists() else None
        with open(agnostic_path, encoding="utf-8") as f:
            raw_tokens = f.read().split()
        label = [SYM_TO_IDX.get(t, SYM_TO_IDX["<unk>"]) for t in raw_tokens]
        if not label:
            continue
        by_stem[agnostic_path.stem].append((str(img_path), distorted_path, label))
    return by_stem


combined_by_stem = defaultdict(list)
for stem, items in _collect_samples_by_stem(CANARY_ROOT).items():
    combined_by_stem[stem].extend(items)
for font in FONTS:
    for stem, items in _collect_samples_by_stem(FONT_DATA_ROOT / font).items():
        combined_by_stem[stem].extend(items)

print(f"{len(combined_by_stem)} unique stems, "
      f"{sum(len(v) for v in combined_by_stem.values())} total (image, label) rows "
      f"before excluding canary stems (fonts included: {FONTS}, LOO_FONT={LOO_FONT!r})")

# Canary stems (base run's original 10% val split) must never be trained on here --
# they're the forgetting probe and need to stay a clean, untouched held-out set.
CANARY_VAL_STEMS = {Path(_canary_clean_full.samples[i][0]).stem for i in CANARY_VAL_INDICES}
combined_by_stem = {s: v for s, v in combined_by_stem.items() if s not in CANARY_VAL_STEMS}
print(f"{len(CANARY_VAL_STEMS)} canary stems excluded -> {len(combined_by_stem)} stems remain")

# -- Hard invariant checks (2026-09-16 fix) --------------------------------------------
# BASE_TRAIN_STEMS is the base model's own training stems (cell 11). Previously the
# equivalence combined_by_stem.keys() == BASE_TRAIN_STEMS was only ever OBSERVED via a
# print, not enforced -- it held only because every source dir happens to share the same
# underlying stem set today. If a font ever fails to render a subset of stems, or
# CANARY_ROOT ever contains stems outside the original split, this would silently drift
# with no code path noticing. Asserting it here turns a latent bug into a loud failure.
BASE_TRAIN_STEMS = {Path(_canary_clean_full.samples[i][0]).stem for i in BASE_TRAIN_INDICES}
assert set(combined_by_stem.keys()) <= BASE_TRAIN_STEMS, (
    "combined_by_stem contains stems outside BASE_TRAIN_STEMS after canary exclusion -- "
    "the assumed equivalence has drifted (see cell docstring); investigate before "
    "trusting any split derived from this pool.")

# -- Stem-based 90/10 split ------------------------------------------------------------
rng = random.Random(42)
all_stems = sorted(combined_by_stem.keys())
rng.shuffle(all_stems)
_val_n = int(len(all_stems) * 0.1)
finetune_val_stems   = set(all_stems[:_val_n])
finetune_train_stems = set(all_stems[_val_n:])

assert finetune_val_stems <= BASE_TRAIN_STEMS
assert finetune_val_stems.isdisjoint(finetune_train_stems)

finetune_train_samples = [s for stem in finetune_train_stems for s in combined_by_stem[stem]]
finetune_val_samples   = [s for stem in finetune_val_stems   for s in combined_by_stem[stem]]
print(f"Fine-tune train: {len(finetune_train_stems)} stems / {len(finetune_train_samples)} rows")
print(f"Fine-tune val:   {len(finetune_val_stems)} stems / {len(finetune_val_samples)} rows")

# -- Caveat (relabeled 2026-09-16, see docs/decisions/piya-106-crnn-font-retraining-plan.md):
# font_val_ser is a content-memorization-confounded SANITY CHECK ("fine-tuning didn't break
# decoding under a new font, on already-known content"), NOT a font-generalization metric --
# every stem in finetune_val_stems is a subset of BASE_TRAIN_STEMS by construction (the
# assertion above), so this can never measure performance on content the base model hasn't
# already seen (in the original font). Report it only as a collapse/divergence check.
_overlap = finetune_val_stems & BASE_TRAIN_STEMS
print(f"\n{len(_overlap)}/{len(finetune_val_stems)} font-val stems "
      f"({len(_overlap) / len(finetune_val_stems) * 100:.1f}%) were already seen by the "
      f"base model during its original CameraPrIMuS training (in the original font) -- "
      f"expected to be ~100% by construction (see assertion above). font_val_ser is a "
      f"memorization-confounded sanity check, NOT font-generalization evidence; see "
      f"loo_font_ser below for the actual (partial) generalization signal.")

# -- Leave-one-font-out (LOFO) probe: LOO_FONT never appears in the training pool above,
# for any stem, in any split -- so loo_font_ser (computed in Cell C) is the closest thing
# this plan can offer to a genuine font-generalization number without new data collection.
# Still content the base model has seen (in the original font), so it isolates GLYPH-STYLE
# generalization specifically, not full novel-content generalization -- report it as that,
# not conflated with BUG-011's real-screenshot eval (Step 3/4 of the plan) which remains
# the actual go/no-go gate.
loo_font_by_stem = _collect_samples_by_stem(FONT_DATA_ROOT / LOO_FONT)
loo_font_by_stem = {s: v for s, v in loo_font_by_stem.items() if s not in CANARY_VAL_STEMS}
assert set(loo_font_by_stem.keys()) <= BASE_TRAIN_STEMS
loo_font_samples = [s for stem in loo_font_by_stem for s in loo_font_by_stem[stem]]
print(f"LOO font ({LOO_FONT}): {len(loo_font_by_stem)} stems / {len(loo_font_samples)} rows "
      f"-- held out of training entirely, eval-only")


class CombinedFineTuneDataset(CameraPrIMuSDataset):
    """Reuses CameraPrIMuSDataset's image-loading + _augment pipeline unchanged, but is
    built from a pre-assembled `samples` list spanning multiple source directories
    instead of globbing a single root, so samples can be split by stem across sources
    before construction (see stem-based split above)."""

    def __init__(self, samples, img_height=64, augment=False,
                 distorted_prob=0.5, force_distorted=False):
        self.vocab = SYM_TO_IDX
        self.img_height = img_height
        self.augment = augment
        self.distorted_prob = distorted_prob
        self.force_distorted = force_distorted
        self.samples = samples
        n_distorted = sum(1 for _, d, _ in self.samples if d is not None)
        print(f"CombinedFineTuneDataset: {len(self.samples)} samples "
              f"({n_distorted} have a _distorted.jpg variant)")


finetune_train_ds = CombinedFineTuneDataset(finetune_train_samples, img_height=64,
                                             augment=True, distorted_prob=0.5)
finetune_val_ds   = CombinedFineTuneDataset(finetune_val_samples, img_height=64, augment=False)
loo_font_ds        = CombinedFineTuneDataset(loo_font_samples, img_height=64, augment=False)

finetune_train_loader = DataLoader(finetune_train_ds, batch_size=16, shuffle=True,
                                    collate_fn=ctc_collate, num_workers=0, pin_memory=True)
finetune_val_loader   = DataLoader(finetune_val_ds, batch_size=16, shuffle=False,
                                    collate_fn=ctc_collate, num_workers=0, pin_memory=True)
loo_font_loader       = DataLoader(loo_font_ds, batch_size=16, shuffle=False,
                                    collate_fn=ctc_collate, num_workers=0, pin_memory=True)

In [ ]:
# Step 2, Cell C — fine-tuning loop with a two-tier canary gate (amended 2026-09-15, see
# docs/decisions/piya-106-crnn-font-retraining-plan.md): the tight bootstrap-CI floor from
# Cell A is DIAGNOSTIC ONLY (logged every epoch, never stops a run). The sole enforced
# auto-stop is a looser HARD CAP, an explicit human-picked multiplier on each canary
# subset's own baseline mean -- catches genuine runaway forgetting without tripping on
# ordinary SGD noise from a single real gradient update. Every epoch's checkpoint now
# saves unconditionally (tagged with gate_mode + the thresholds in effect), so a full
# clean/distorted-SER trend is always recoverable, even from a run that never beats its
# own "best font-val SER so far" criterion -- fixes the first sweep producing zero saved
# weights when the old zero-tolerance gate broke at epoch 1 for every LR.
#
# loo_font_loader (added 2026-09-16, see docs/decisions/piya-106-crnn-font-retraining-plan.md):
# logged-only diagnostic, same as font_val_ser -- does not gate anything. Unlike
# font_val_ser (memorization-confounded, see cell 12), loo_font_ser is measured on a font
# LOO_FONT held out of training entirely, so it's the closest thing this plan offers to a
# genuine (if content-familiar) font-generalization signal without new data collection.

GATE_MODE = "soft_diagnostic_hard_cap_v1"


def finetune(model, train_loader, val_loader,
             canary_clean_loader, canary_distorted_loader,
             canary_clean_hard_cap, canary_distorted_hard_cap,
             device, weights_out="crnn_weights/finetuned_font.pt",
             max_epochs=20, lr=3e-5, patience=3,
             canary_clean_soft_floor=None, canary_distorted_soft_floor=None,
             loo_font_loader=None, seed=None):
    # Seed replication (2026-09-16, ml-cv-lead follow-up on the extended trend check): the
    # same (lr, epoch-1) result swung ~83% relative between two runs with no explicit seed
    # control (e.g. lr=1e-5 epoch 1 loo_font_ser: 0.0608 vs 0.1111) -- bigger than the gap
    # currently used to rank LRs against each other. Seeding here makes a given (lr, seed)
    # run reproducible, and running the same lr under two different seeds is how that
    # variance gets measured instead of silently confounding LR comparisons.
    # Hard requirement (2026-09-16, ml-cv-lead review of the real RUN_FINETUNING run):
    # an omitted seed does NOT draw fresh randomness -- it inherits whatever RNG state
    # was left over from the last random-consuming call in this kernel (e.g. a prior
    # cell execution's seed=2 sub-run), making an "unseeded" run a deterministic replay
    # keyed on execution history, not an independent sample. Root-caused after two
    # separate unseeded runs (0.0485, then 0.0398) both beat every seeded run in this
    # plan -- consistent with leftover state, not luck. Failing loudly here is cheaper
    # than silently trusting a non-reproducible number again.
    assert seed is not None, (
        "finetune() requires an explicit seed -- see docstring above; do not rely on "
        "whatever RNG state the kernel happens to be in.")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # GPU-level determinism (2026-09-16, seed=1 isolated reproduction check): torch.manual_seed
    # alone does not make CUDA/cuDNN convolution kernel selection reproducible -- cuDNN can pick
    # different (nondeterministic) algorithms run to run, which is independent of the RNG seed
    # above. Confirmed empirically: the same seed=1 config produced a clean 4-epoch trend on one
    # run and a hard-cap breach by epoch 2 on a rerun with identical seed/data/hyperparameters.
    # These two lines force deterministic cuDNN kernels so a given seed is actually reproducible.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    criterion = torch.nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    best_selection_ser = float("inf")
    best_epoch = None
    epochs_since_improve = 0
    os.makedirs(os.path.dirname(weights_out) or ".", exist_ok=True)
    weights_stem, weights_ext = os.path.splitext(weights_out)

    epoch_history = []  # (epoch, canary_clean_ser, canary_distorted_ser) for the epoch-2 check-in

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for images, labels, in_lens, lb_lens in train_loader:
            images, labels = images.to(device), labels.to(device)
            in_lens, lb_lens = in_lens.to(device), lb_lens.to(device)
            log_probs = model(images)
            in_lens = in_lens.clamp(max=log_probs.shape[0])
            loss = criterion(log_probs, labels, in_lens, lb_lens)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        font_val_ser         = _evaluate_ser(model, val_loader, device)
        canary_clean_ser     = _evaluate_ser(model, canary_clean_loader, device)
        canary_distorted_ser = _evaluate_ser(model, canary_distorted_loader, device)
        loo_font_ser = (_evaluate_ser(model, loo_font_loader, device)
                         if loo_font_loader is not None else float("nan"))
        epoch_time = time.time() - t0
        epoch_history.append((epoch, canary_clean_ser, canary_distorted_ser))

        # Selection metric (2026-09-16, ml-cv-lead review of the first real-data sweep):
        # font_val_ser is confounded (see cell 12 -- its val split is a subset of the base
        # model's own training stems), so "best epoch" picked on it can favor a checkpoint
        # that only got better at recalling already-memorized content. loo_font_ser is
        # measured on a font held out of training entirely, so it has actual construct
        # validity for font-domain generalization -- use it for both best-checkpoint
        # selection and early-stopping patience whenever a LOO loader is available,
        # falling back to font_val_ser only when one isn't (e.g. a caller that never
        # wired up loo_font_loader).
        selection_ser = loo_font_ser if loo_font_loader is not None else font_val_ser
        selection_metric_name = "loo_font_ser" if loo_font_loader is not None else "font_val_ser"

        hard_cap_ok = (canary_clean_ser <= canary_clean_hard_cap and
                       canary_distorted_ser <= canary_distorted_hard_cap)
        soft_floor_breach = (
            (canary_clean_soft_floor is not None and canary_clean_ser > canary_clean_soft_floor) or
            (canary_distorted_soft_floor is not None and canary_distorted_ser > canary_distorted_soft_floor))

        if not hard_cap_ok:
            status = "HARD CAP BREACHED"
        elif soft_floor_breach:
            status = "soft floor breached (diagnostic only)"
        else:
            status = "OK"

        print(f"Epoch {epoch:03d} | loss {train_loss/len(train_loader):.4f} | "
              f"font-val SER {font_val_ser:.4f} (memorization-confounded, see cell 12) | "
              f"LOO-font SER {loo_font_ser:.4f} (genuine font-generalization signal) | "
              f"canary clean {canary_clean_ser:.4f} (hard cap {canary_clean_hard_cap:.4f}) | "
              f"canary distorted {canary_distorted_ser:.4f} (hard cap {canary_distorted_hard_cap:.4f}) | "
              f"{status} | {epoch_time:.1f}s")

        # Per-epoch checkpoint, unconditional -- see cell docstring above for why.
        epoch_ckpt_path = f"{weights_stem}_epoch{epoch}{weights_ext}"
        torch.save({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "epoch": epoch,
            "font_val_ser": font_val_ser,
            "loo_font_ser": loo_font_ser,
            "canary_clean_ser": canary_clean_ser,
            "canary_distorted_ser": canary_distorted_ser,
            "gate_mode": GATE_MODE,
            "canary_clean_hard_cap": canary_clean_hard_cap,
            "canary_distorted_hard_cap": canary_distorted_hard_cap,
            "canary_clean_soft_floor": canary_clean_soft_floor,
            "canary_distorted_soft_floor": canary_distorted_soft_floor,
            "vocab": VOCAB,
            "finetune_val_stems": sorted(finetune_val_stems),
            "canary_val_stems": sorted(CANARY_VAL_STEMS),
        }, epoch_ckpt_path)

        if epoch == 2:
            e1_clean, e1_dist = epoch_history[0][1], epoch_history[0][2]
            print(f"  >>> MANUAL CHECK-IN (epoch 2): clean canary delta vs epoch 1 "
                  f"{canary_clean_ser - e1_clean:+.4f}, distorted delta "
                  f"{canary_distorted_ser - e1_dist:+.4f}. If clean is still rising, consider "
                  f"interrupting the kernel before epoch 3 rather than spending compute on a "
                  f"trend that's already clear.")

        if not hard_cap_ok:
            print(f"  Stopping: canary HARD CAP breached at epoch {epoch}. Best checkpoint "
                  f"remains epoch {best_epoch} ({selection_metric_name} {best_selection_ser:.4f}). "
                  f"Per-epoch checkpoints up to this point were still saved above.")
            break

        if selection_ser < best_selection_ser:
            best_selection_ser = selection_ser
            best_epoch = epoch
            epochs_since_improve = 0
            torch.save({
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "epoch": epoch,
                "font_val_ser": font_val_ser,
                "loo_font_ser": loo_font_ser,
                "canary_clean_ser": canary_clean_ser,
                "canary_distorted_ser": canary_distorted_ser,
                "gate_mode": GATE_MODE,
                "canary_clean_hard_cap": canary_clean_hard_cap,
                "canary_distorted_hard_cap": canary_distorted_hard_cap,
                "canary_clean_soft_floor": canary_clean_soft_floor,
                "canary_distorted_soft_floor": canary_distorted_soft_floor,
                "vocab": VOCAB,
                # Persisted by stem identity (not position) so a future session can
                # recover this exact split without depending on glob-order determinism --
                # the gap that made the base checkpoint's val set unrecoverable (see
                # cell-11 note above).
                "finetune_val_stems": sorted(finetune_val_stems),
                "canary_val_stems": sorted(CANARY_VAL_STEMS),
            }, weights_out)
            print(f"  Saved best fine-tuned model ({selection_metric_name}={selection_ser:.4f}) -> {weights_out}")
        else:
            epochs_since_improve += 1
            if epochs_since_improve >= patience:
                print(f"  Stopping: no {selection_metric_name} improvement for {patience} epochs.")
                break

    print(f"\nFine-tuning complete. Best epoch: {best_epoch}, best {selection_metric_name}: {best_selection_ser:.4f}")
    return best_epoch, best_selection_ser


# -- LR sweep -- short (3-epoch) budget per candidate, each starting fresh from the base
# checkpoint (not cumulative), so the LR is picked by data instead of a priori.
def lr_sweep(lrs, sweep_epochs=3, patience=None, seed=None):
    # patience=None keeps the old "never early-stop within the sweep budget" behavior
    # (patience == sweep_epochs); pass an explicit smaller value (ml-cv-lead recommended
    # ~2, keyed on loo_font_ser) for a longer diagnostic run where early-stopping should
    # actually engage instead of always running the full budget.
    # seed=None runs with whatever RNG state happens to be live (the old, uncontrolled
    # behavior); pass an explicit int to make a run reproducible and comparable across
    # LRs/reruns (see finetune()'s docstring on why this matters).
    results = {}
    for lr in lrs:
        print(f"\n=== LR sweep: lr={lr}{f', seed={seed}' if seed is not None else ''} ===")
        sweep_model = CRNN(VOCAB_SIZE).to(device)
        sweep_model.load_state_dict(_base_ckpt["model_state"])
        _, best_ser = finetune(
            sweep_model, finetune_train_loader, finetune_val_loader,
            canary_clean_loader, canary_distorted_loader,
            CANARY_CLEAN_HARD_CAP_SER, CANARY_DISTORTED_HARD_CAP_SER,
            device, weights_out=f"crnn_weights/sweep_lr{lr}.pt",
            max_epochs=sweep_epochs, lr=lr, patience=(patience if patience is not None else sweep_epochs),
            canary_clean_soft_floor=CANARY_CLEAN_FLOOR_SER,
            canary_distorted_soft_floor=CANARY_DISTORTED_FLOOR_SER,
            loo_font_loader=loo_font_loader, seed=seed)
        results[lr] = best_ser
    print("\nLR sweep results (lower loo_font_ser is better; falls back to font_val_ser "
          "only if no LOO loader was passed):")
    for lr, ser in sorted(results.items(), key=lambda kv: kv[1]):
        print(f"  lr={lr}: best selection SER {ser:.4f}")
    return results


RUN_LR_SWEEP = False  # already run against real font-augmented data on 2026-09-16; see
                       # docs/decisions/piya-106-crnn-font-retraining-plan.md for the result
                       # table and the ml-cv-lead review that followed it
if RUN_LR_SWEEP:
    lr_sweep([1e-5, 3e-5, 1e-4])

# -- Extended trend check (2026-09-16, ml-cv-lead review of the first real-data 3-epoch
# sweep): that sweep picked its "best epoch" on font_val_ser, which is confounded (cell 12),
# and only ran 3 epochs per LR with n=1 -- not enough to tell a real trend from noise,
# especially given the observed non-monotonicity (e.g. lr=3e-5 breach -> pass -> breach).
# lr=1e-4 is excluded here: its epoch-1 result was the sweep's best raw score but immediately
# preceded the steepest forgetting slope observed (canary clean landed within ~7% of the
# hard cap by epoch 3) -- a cliff-edge result, not a stable optimum worth extending.
# lr=3e-5 is the lead candidate (epoch 2 was the only point where canary genuinely improved
# over baseline in both clean and distorted, with the best loo_font_ser after excluding
# lr=1e-4); lr=1e-5 is the backup. Selection now runs on loo_font_ser (see finetune() above),
# with patience=2 so early stopping can actually engage instead of always spending the full
# epoch budget. Confirm Kaggle GPU quota headroom before running -- this is 2 LRs x up to 8
# epochs, notably more compute than the first sweep.
RUN_EXTENDED_TREND_CHECK = False  # already run 2026-09-16: lr=3e-5 best at epoch 1
                                   # (loo_font_ser 0.0485), lr=1e-5 best at epoch 3 (0.0723);
                                   # see docs/decisions/piya-106-crnn-font-retraining-plan.md
if RUN_EXTENDED_TREND_CHECK:
    lr_sweep([3e-5, 1e-5], sweep_epochs=8, patience=2)

# -- Seed replication check (2026-09-16, ml-cv-lead follow-up): the extended trend check
# above ran with no explicit seed control, and the SAME (lr, epoch-1) result swung ~83%
# relative between the first 3-epoch sweep and this rerun (lr=1e-5 epoch 1 loo_font_ser:
# 0.0608 vs 0.1111) -- bigger than the gap being used to rank LRs against each other. Before
# treating lr=3e-5 epoch 1's strong result (loo_font_ser 0.0485, canary fully clean) as a
# real candidate, replicate it under controlled seeds and check the spread. Short budget
# (2 epochs) is enough to check epoch-1 stability without re-spending the full sweep cost.
RUN_SEED_REPLICATION_CHECK = False  # flip to True and re-run this cell for the replication check
if RUN_SEED_REPLICATION_CHECK:
    for _seed in (1, 2):
        print(f"\n### Seed replication check: seed={_seed} ###")
        lr_sweep([3e-5, 1e-5], sweep_epochs=2, patience=2, seed=_seed)

# -- Full fine-tuning run -- set LR from the sweep above (or leave the 3e-5 default,
# 10x below the base run's 3e-4, if skipping the sweep) before flipping this flag.
#
# Seed replication check result (2026-09-16, ml-cv-lead review): lr=3e-5 beat lr=1e-5
# in both seeds (0.0540 vs 0.0702 at seed=1; 0.0696 vs 0.0924 at seed=2), a consistent
# ~24% relative margin -- real signal, not a fluke. But the earlier 0.0485 headline
# number was an uncontrolled-seed outlier below both seeded runs; treat 0.054-0.073 as
# the realistic range instead. Two open risks this run inherits: (1) loo_font_ser is
# itself noisy -- lr=1e-5 epoch-1 still swung ~61% relative between seeds even under
# explicit seeding (0.0702 vs 0.1132), so single-epoch deltas under ~0.02 aren't
# trustworthy signal; (2) epoch-of-best flips direction between seeds for lr=3e-5
# (seed=1 peaked at epoch 1 then dropped 81% relative by epoch 2; seed=2 was still
# improving at epoch 2), and neither seed has epoch-3+ data, so patience is raised
# from 2 to 3 here (vs. the seed-replication check's own patience=2) to give one more
# epoch of runway to distinguish a recoverable dip from real decline before stopping.
# Whatever loo_font_ser the selected checkpoint reports, carry it with a ~+/-0.02-0.03
# error-band caveat -- it is a single-run estimate on a small seed sample, not a clean
# point number -- before it goes into docs/decisions/ or the bug tracker.
#
# Two-seed final run (2026-09-16, ml-cv-lead + root-cause finding): the first
# RUN_FINETUNING attempt omitted seed= and produced loo_font_ser=0.0398 at epoch 1 --
# lower than every seeded run so far. Root cause: finetune() only touches RNG state
# when seed is not None (see its assert above, added because of this), so an unseeded
# call in an already-running kernel inherits whatever RNG state the PREVIOUS cell
# execution left behind (here, the tail end of the seed-replication check's
# seed=2/lr=1e-5 sub-run) instead of drawing fresh randomness. That made the 0.0398
# result a deterministic replay keyed on execution history, not an independent
# sample -- not trustworthy as the plan's reported number, and that checkpoint
# (crnn_weights/finetuned_font.pt) should not be used downstream.
#
# finetune() now hard-fails without an explicit seed, so this reruns the real
# fine-tune under BOTH validated seeds (1 and 2, same ones from the seed-replication
# check) at lr=3e-5/patience=3/max_epochs=6, each from a fresh copy of the base
# checkpoint (mirrors lr_sweep()'s pattern -- seed=2 must not train on top of
# seed=1's weights). Report the resulting loo_font_ser as this 2-seed distribution
# (with the +/-0.02-0.03 caveat above), not a single cherry-picked number.
RUN_FINETUNING = True  # flip to True and re-run this cell to actually fine-tune
if RUN_FINETUNING:
    final_results = {}
    for _seed in (1, 2):
        print(f"\n### Final fine-tune: seed={_seed} ###")
        final_model = CRNN(VOCAB_SIZE).to(device)
        final_model.load_state_dict(_base_ckpt["model_state"])
        _, best_ser = finetune(
            final_model, finetune_train_loader, finetune_val_loader,
            canary_clean_loader, canary_distorted_loader,
            CANARY_CLEAN_HARD_CAP_SER, CANARY_DISTORTED_HARD_CAP_SER,
            device, weights_out=f"crnn_weights/finetuned_font_seed{_seed}.pt",
            max_epochs=6, lr=3e-5, patience=3,
            canary_clean_soft_floor=CANARY_CLEAN_FLOOR_SER,
            canary_distorted_soft_floor=CANARY_DISTORTED_FLOOR_SER,
            loo_font_loader=loo_font_loader, seed=_seed)
        final_results[_seed] = best_ser
    print("\nFinal fine-tune results by seed (lower loo_font_ser is better):")
    for _seed, ser in sorted(final_results.items(), key=lambda kv: kv[1]):
        print(f"  seed={_seed}: best loo_font_ser {ser:.4f}")

In [ ]:
# Diagnostic-only seed=1 reproduction check -- already run twice and documented in
# docs/decisions/piya-106-crnn-font-retraining-plan.md (2026-09-19 amendment). Off by
# default so a real RUN_FINETUNING pass (cell above) does not also re-spend ~30 min of
# GPU time re-confirming a result that is already recorded. Flip to True only if you
# specifically need a third reproduction data point.
RUN_REPRO_CHECK = False
if RUN_REPRO_CHECK:
    repro_model = CRNN(VOCAB_SIZE).to(device)
    repro_model.load_state_dict(_base_ckpt["model_state"])
    finetune(repro_model, finetune_train_loader, finetune_val_loader,
             canary_clean_loader, canary_distorted_loader,
             CANARY_CLEAN_HARD_CAP_SER, CANARY_DISTORTED_HARD_CAP_SER,
             device, weights_out="crnn_weights/repro_check_seed1.pt",
             max_epochs=6, lr=3e-5, patience=3,
             canary_clean_soft_floor=CANARY_CLEAN_FLOOR_SER,
             canary_distorted_soft_floor=CANARY_DISTORTED_FLOOR_SER,
             loo_font_loader=loo_font_loader, seed=1)